# Onset-HFO 5 — Did the HFO map point at the tissue whose removal cured the patient?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/berdakh/onset-hfo/blob/master/onset-hfo/notebooks/05_surgical_outcome_study.ipynb)

Every other notebook here compares an algorithm to another algorithm, or to a
human reading the same screen. This one compares it to **what happened to the
patient after surgery** — the only reference standard in epilepsy surgery that
is not another opinion.

**The dataset makes it possible.** OpenNeuro `ds003498` (Zurich, 20 patients,
interictal slow-wave sleep, 2000 Hz) ships three things in the same archive:
expert HFO markings per channel, the **resected contacts** for each patient,
and whether that patient became **seizure-free**. Almost no public iEEG
dataset carries all three.

**What you will do.** Reproduce a published finding from 60 seconds of
recording per patient, then watch our own detector fail to reach it — which is
the more useful of the two results, because it says precisely what to fix.

**Runtime.** About 25 minutes on a free Colab instance, most of it downloading
20 x ~24 MB slices. Everything is cached, so a second run is compute-only.

In [1]:
# Colab setup. On your own machine, skip this cell and run
#   pip install -e ".[dev]"  from the onset-hfo directory instead.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/berdakh/onset-hfo.git"
BRANCH = "master"   # change to the default branch once this work is merged

if IN_COLAB and not os.path.exists("onset-hfo"):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, "--depth", "1", REPO], check=True)
if IN_COLAB:
    os.chdir("/content/onset-hfo/onset-hfo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

working directory: /home/user/bci-gan/onset-hfo


---
## 1. The three ingredients

Read them straight out of the archive before running anything, so you can see
what the study is actually made of.

In [2]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from onset_hfo.clinical import fetch_participants, resection_map

participants = fetch_participants("ds003498")
print(participants[["subject", "epilepsy", "ilae", "outcome",
                    "months_follow_up"]].to_string(index=False))
print("\nseizure-free (S) vs recurrence (F):",
      participants["outcome"].value_counts().to_dict())

subject epilepsy  ilae outcome  months_follow_up
 sub-01      TLE     1       S                12
 sub-02      TLE     1       S                29
 sub-03      TLE     1       S                13
 sub-04      TLE     1       S                41
 sub-05      TLE     1       S                14
 sub-06      TLE     1       S                11
 sub-07      TLE     3       F                42
 sub-08      TLE     3       F                15
 sub-09      TLE     5       F                46
 sub-10      ETE     1       S                36
 sub-11      ETE     1       S                37
 sub-12      ETE     1       S                25
 sub-13      ETE     1       S                25
 sub-14      ETE     1       S                10
 sub-15      ETE     1       S                25
 sub-16      ETE     1       S                19
 sub-17      ETE     5       F                45
 sub-18      ETE     5       F                30
 sub-19      ETE     6       F                11
 sub-20      ETE    

`S` means **success** — seizure free. `F` means **failure** — seizures
returned. Reading `F` as "free" inverts every label in the cohort, which is
why `onset_hfo.clinical` never exposes the raw letter without the mapping
beside it.

Now the resected zone. The archive stores it as a clinician's free text, one
row per patient:

In [3]:
resections = resection_map("ds003498")
for subject in ["sub-01", "sub-12", "sub-15"]:
    r = resections[subject]
    print(f"{subject}: rz text {r.rz_text!r}")
    print(f"          -> {len(r.resected)} contacts: {', '.join(r.resected[:8])}"
          + (" ..." if len(r.resected) > 8 else ""))
    if r.eloquent:
        print(f"          eloquent (excluded by the source study): {', '.join(r.eloquent)}")
    if r.unparsed:
        print(f"          COULD NOT PARSE: {r.unparsed}  <- reported, not silently dropped")

[onset-hfo] sub-15: could not parse '1ll22-24' in the clinical sheet (left out of the contact lists)
sub-01: rz text 'ahr1-4, ar1-4, phr1-4'
          -> 12 contacts: AHR1, AHR2, AHR3, AHR4, AR1, AR2, AR3, AR4 ...
sub-12: rz text 'gl1-3, gl9-14, gl17-22, gl25-32, tl1-4'
          -> 27 contacts: GL1, GL2, GL3, GL9, GL10, GL11, GL12, GL13 ...
sub-15: rz text 'tbal2-4, tll1-2, tll9-10'
          -> 7 contacts: TBAL2, TBAL3, TBAL4, TLL1, TLL2, TLL9, TLL10
          eloquent (excluded by the source study): TLL6, TLL7, TLL8, TLL14, TLL15, TLL16, TLL31, TLL32
          COULD NOT PARSE: ('1ll22-24',)  <- reported, not silently dropped


That last line is the point of the `unparsed` field. Subject 15's sheet says
`1ll22-24` where every other token on the row says `tll` — a typo in the
original clinical data. A parser that quietly returned the other three ranges
would shrink that patient's excluded set with nothing to show for it.

## 2. Contacts are not channels

The sheet names **contacts** (`AHR1`). The analysis runs on **bipolar
channels** (`AHR1-AHR2`), each spanning two contacts. So a channel is inside
the resection, outside it, or — at the margin — straddling it:

In [4]:
from onset_hfo.clinical import classify_channels
from onset_hfo.datasets import fetch_slice
from onset_hfo.preprocess import prepare

rec = fetch_slice(dataset="ds003498", subject="sub-01", run="01",
                  t_start=0, t_stop=60, verbose=False)
prep = prepare(rec, verbose=False)
labels = classify_channels(prep.ch_names, resections["sub-01"])

print(labels["zone"].value_counts().to_string())
print("\nthe resection margin (one contact in, one out):")
print(labels[labels["zone"] == "partial"][["channel", "contact_a", "contact_b"]]
      .to_string(index=False))
print("\nhow much of the resection this recording can even see:")
print(resections["sub-01"].coverage(prep.ch_names))

zone
spared      31
resected     9
partial      3

the resection margin (one contact in, one out):
  channel contact_a contact_b
AHR4-AHR5      AHR4      AHR5
  AR4-AR5       AR4       AR5
PHR4-PHR5      PHR4      PHR5

how much of the resection this recording can even see:
{'subject': 'sub-01', 'rz_contacts_listed': 12, 'rz_contacts_recorded': 12, 'rz_coverage': 1.0, 'missing': ''}


**`partial` is kept as its own label on purpose.** Folding margin channels
into "resected" inflates every "we got it all" number; folding them into
"spared" inflates the opposite one. In the primary metric they sit in the
denominator and not the numerator, so a detector firing along the margin gets
no credit for it.

**`rz_coverage` is the honest denominator.** In five of the twenty patients —
all temporal-lobe cases, where the source study kept only the three most
mesial bipolar channels — only 4 of 16 resected contacts were recorded at all.
Their "share inside the resection" describes a quarter of a resection, and the
study writes that number to `recordings.csv` rather than burying it.

## 3. The design, before any numbers

Four choices, each of which can only make the result *worse*:

| Choice | Why |
|---|---|
| **An expert positive control** | Every number is computed twice — from the published expert markings and from our detector, on the same channels. Without it, a null is unreadable: it could be the detector or it could be 60 seconds and 20 patients. |
| **Operating points fixed in advance** | 2.0 SD for ripples, 5.0 SD for fast ripples, both chosen in `benchmark.py` on *channel-rank agreement with the experts* — a question that says nothing about surgery. Tuning a threshold against outcome and then reporting the outcome would be circular. |
| **Margin channels not claimed** | as above. |
| **Both channel scopes reported** | `reviewed` (what the annotators marked) and `all` (what a deployed tool would face). Reporting only the flattering one is a choice made after seeing both. |

## 4. Run it

In [5]:
from onset_hfo.outcome import outcome_study

result = outcome_study(verbose=True)   # ~20 minutes cold, ~4 minutes cached
result.save()

[onset-hfo] sub-15: could not parse '1ll22-24' in the clinical sheet (left out of the contact lists)



[onset-hfo] (1/20) sub-01 [outcome S]
[onset-hfo] using cached slice /home/user/bci-gan/onset-hfo/artifacts/data/ds003498/sub-01_run-01_0-60s


[onset-hfo] loaded sub-01/None/run-01: 50 channels, 60.0 s @ 2000 Hz; seizure none marked


[onset-hfo]        expert: 0.34 of 752 events inside the resection (9/23 reviewed channels resected)
[onset-hfo]           rms: 0.51 of 899 events inside the resection (9/23 reviewed channels resected)

[onset-hfo] (2/20) sub-02 [outcome S]
[onset-hfo] using cached slice /home/user/bci-gan/onset-hfo/artifacts/data/ds003498/sub-02_run-01_0-60s


[onset-hfo] loaded sub-02/None/run-01: 64 channels, 60.0 s @ 2000 Hz; seizure none marked


...

[onset-hfo]        expert: 0.10 of 2162 events inside the resection (6/42 reviewed channels resected)
[onset-hfo]           rms: 0.10 of 2030 events inside the resection (6/42 reviewed channels resected)

[onset-hfo] (20/20) sub-20 [outcome F]
[onset-hfo] using cached slice /home/user/bci-gan/onset-hfo/artifacts/data/ds003498/sub-20_run-01_0-60s


[onset-hfo] loaded sub-20/None/run-01: 16 channels, 60.0 s @ 2000 Hz; seizure none marked


[onset-hfo]        expert: 0.30 of 893 events inside the resection (5/12 reviewed channels resected)
[onset-hfo]           rms: 0.36 of 654 events inside the resection (5/12 reviewed channels resected)


[onset-hfo] outcome study written to /home/user/bci-gan/onset-hfo/artifacts/results/outcome_ds003498


PosixPath('/home/user/bci-gan/onset-hfo/artifacts/results/outcome_ds003498')

## 5. The metric that carries the signal

Four metrics were computed. They disagree, and the disagreement is itself a
finding.

In [6]:
pd.set_option("display.width", 220)
print("--- was the single busiest channel resected? ---")
print(result.summary("top_channel_resected")
      .query("scope == 'reviewed'").to_string(index=False))
print("\n--- what fraction of all HFO events was inside the resection? ---")
print(result.summary("share_in_rz")
      .query("scope == 'reviewed'").to_string(index=False))

--- was the single busiest channel resected? ---
source    scope        band  n_seizure_free  n_recurrence  median_seizure_free  median_recurrence  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  rank_biserial  p_permutation  p_bonferroni
expert reviewed fast_ripple              13             7                  1.0                0.0              0.923            0.286 0.819   0.637   1.000          0.637          0.007         0.173
expert reviewed      ripple              13             7                  0.0                0.0              0.385            0.000 0.692   0.577   0.846          0.385          0.114         1.000
   rms reviewed fast_ripple              12             7                  1.0                0.0              0.833            0.429 0.702   0.488   0.917          0.405          0.129         1.000
   rms reviewed      ripple              13             7                  0.0                0.0              0.462            0.000 0.731   0.615   0

Read the `expert` / `fast_ripple` row of the first table first. The busiest
fast-ripple channel was inside the resection in **12 of 13** patients who
became seizure-free and **2 of 7** whose seizures returned — AUC 0.82,
permutation p = 0.007. That is the published claim of Fedele et al. 2017, the
study this dataset comes from, reproduced from one minute of recording per
patient.

Then read the same row of the second table: `share_in_rz` shows **nothing**,
in the expert arm, on the same events. **Concentration localises; proportion
does not.** "Most of this patient's HFOs were inside the resection" is largely
a statement about how big the resection was. "The one place generating the
most fast ripples was removed" is the clinically useful sentence — and it
means any report built on this pipeline should show a *ranking*, not a
percentage.

## 6. Where our detector stands

In [7]:
print(result.verdict())          # the pre-specified comparison
print()
for key, text in result.verdicts().items():
    if key.startswith("top_channel"):
        print(f"{key}:\n  {text}\n")

expert: AUC 0.82 (95% CI 0.64-1.00), p = 0.007 (24-comparison Bonferroni 0.17) -- separates the groups. rms: AUC 0.70 (95% CI 0.49-0.92), p = 0.129 (24-comparison Bonferroni 1.00) -- does not separate the groups. With 13 seizure-free and 7 recurrence patients, the smallest effect reaching 80% power is AUC 0.85; anything below that could not have been detected here whether or not it is real. The expert markings do separate the groups on the same channels, so any failure of the detector arm is the detector's.

top_channel_resected/fast_ripple:
  expert: AUC 0.82 (95% CI 0.64-1.00), p = 0.007 (24-comparison Bonferroni 0.17) -- separates the groups. rms: AUC 0.70 (95% CI 0.49-0.92), p = 0.129 (24-comparison Bonferroni 1.00) -- does not separate the groups. With 13 seizure-free and 7 recurrence patients, the smallest effect reaching 80% power is AUC 0.85; anything below that could not have been detected here whether or not it is real. The expert markings do separate the groups on the same c

In [8]:
subjects = result.subjects.merge(result.participants[["subject", "outcome"]], on="subject")
view = subjects.query("band == 'fast_ripple' and scope == 'reviewed'")
print(view[["subject", "outcome", "source", "n_events",
            "share_in_rz", "top_channel_resected"]]
      .sort_values(["outcome", "subject", "source"]).to_string(index=False))

subject outcome source  n_events  share_in_rz  top_channel_resected
 sub-07       F expert      15.0     0.400000                   0.0
 sub-07       F    rms       1.0     0.000000                   0.0
 sub-08       F expert      33.0     0.303030                   0.0
 sub-08       F    rms     113.0     0.000000                   0.0
 sub-09       F expert     269.0     0.706320                   1.0
 sub-09       F    rms     114.0     0.824561                   1.0
 sub-17       F expert     566.0     0.800353                   1.0
 sub-17       F    rms     503.0     0.815109                   1.0
 sub-18       F expert     166.0     0.138554                   0.0
 sub-18       F    rms      17.0     0.529412                   1.0
 sub-19       F expert     509.0     0.084479                   0.0
 sub-19       F    rms     231.0     0.164502                   0.0
 sub-20       F expert      80.0     0.262500                   0.0
 sub-20       F    rms      33.0     0.181818   

Two things are visible in that table and in nothing else.

**Our detector points the same way and does not get there** — 10 of 12 versus
3 of 7, AUC 0.70, p = 0.13. This is the one configuration in the whole design
that is evidence against *the detector* rather than against the sample size:
the expert positive control cleared the bar on the same patients and the same
channels. That gap is the most useful number in the repository, because it is
a specific engineering target rather than a vague "needs more validation".

**It is also event-starved.** At 5.0 SD in a 60-second window, five of twenty
subjects yield one or zero fast-ripple detections, and `sub-10` yields none
and drops out of that arm entirely (which is why its `n_seizure_free` reads 12,
not 13). A per-patient statistic computed from a single event is not a
measurement. The source study scored whole nights.

## 7. Why the band matters, demonstrated

The first version of this analysis used 2.0 SD in **both** bands, because that
is what the ripple benchmark prefers. Here is what that does in the fast-ripple
band:

In [9]:
wrong = outcome_study(subjects=[f"sub-{i:02d}" for i in range(1, 21)],
                      threshold_sd=2.0,         # one threshold everywhere
                      bands=("fast_ripple",), verbose=False)
print("2.0 SD in both bands:")
print(wrong.summary("top_channel_resected").query("scope == 'reviewed'").to_string(index=False))
print("\nmeasured per-band operating points:")
print(result.summary("top_channel_resected")
      .query("scope == 'reviewed' and band == 'fast_ripple'").to_string(index=False))

2.0 SD in both bands:
source    scope        band  n_seizure_free  n_recurrence  median_seizure_free  median_recurrence  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  rank_biserial  p_permutation  p_bonferroni
expert reviewed fast_ripple              13             7                  1.0                0.0              0.923            0.286 0.819   0.637   1.000          0.637          0.007         0.079
   rms reviewed fast_ripple              13             7                  0.0                0.0              0.462            0.286 0.588   0.368   0.808          0.176          0.642         1.000

measured per-band operating points:
source    scope        band  n_seizure_free  n_recurrence  median_seizure_free  median_recurrence  mean_seizure_free  mean_recurrence   auc  auc_lo  auc_hi  rank_biserial  p_permutation  p_bonferroni
expert reviewed fast_ripple              13             7                  1.0                0.0              0.923            0.286 0.819  

At 2.0 SD the fast-ripple detector runs at precision 0.086 — a mean of 1,142
detections per 60 s against a mean of 228 expert-marked events — and the
outcome signal disappears. Nothing about the outcome data was used to choose
either threshold; both come from channel-rank agreement with the experts, in
`notebooks/03_validation_and_benchmark.ipynb`. **One threshold for both bands
is a bug, not a simplification.**

## 8. Read this before quoting any of it

- **Thirteen versus seven is a very small study.** `min_detectable_auc(13, 7)`
  returns **0.85** — with these group sizes only a very large separation
  reaches 80% power. A p above 0.05 here means *underpowered*, not *no effect*.
- **Twenty-four comparisons, uncorrected.** The Bonferroni column is in
  `groups.csv`; p = 0.007 becomes p = 0.17 across the table. The expert
  fast-ripple row is worth reporting because it is a *pre-specified
  replication* of the paper the dataset accompanies, not because it survived a
  search.
- **Sixty seconds of one night**, against several whole nights in the source
  study. Try `outcome_study(t_stop=300)`.
- **Retrospective, one centre, one surgical team.** The
  [HFO Trial](https://www.thelancet.com/journals/laneur/article/PIIS1474-4422(22)00311-8/fulltext)
  (Lancet Neurology 2022) tested HFO-guided resection prospectively and did not
  find the benefit retrospective series report. This is a retrospective series.
- **Reproducing a retrospective result is evidence that the analysis is sound,
  not that the clinical claim is.**

`docs/OUTCOME.md` carries the full design, every table, and the list of what
this cannot support.

In [10]:
from onset_hfo.outcome import min_detectable_auc
print("smallest AUC detectable at 80% power with 13 vs 7:",
      min_detectable_auc(13, 7))
print("\nfiles written:")
import pathlib
for f in sorted(pathlib.Path("artifacts/results/outcome_ds003498").iterdir()):
    print("  ", f.name)

smallest AUC detectable at 80% power with 13 vs 7: 0.85

files written:
   channels.csv
   groups.csv
   participants.csv
   recordings.csv
   run.json
   subjects.csv
